In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

In [130]:
df = pd.read_csv("Advertising.csv").iloc[:,1:]
df.head()

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9


In [4]:
# Standardize data
mat = df.iloc[:20,0:4].values
mat_standard = (mat - np.mean(mat, axis = 0))/np.std(mat, axis = 0) # z-score to standardize data
x = mat_standard[:,0:3]
x = np.c_[x, np.ones(len(x))] # add 1's for y-intercept matrix multiplication
y = mat_standard[:,3]
x, y

# Below is dataframe version
#df_standard = pd.DataFrame(mat_standard) # convert numpy array to dataframe
#df_standard.columns = ["TV", "Radio", "Newspaper", "Sales"]
#x = df_standard.iloc[:, 0:3]
#x['b'] = 1
#y = df_standard.iloc[:,3]
#df_standard, x, y

(array([[ 1.35032393,  0.66836934,  0.94254628,  1.        ],
        [-0.91179468,  0.7680268 ,  0.10711541,  1.        ],
        [-1.24453088,  1.20651961,  0.9460128 ,  1.        ],
        [ 0.3923362 ,  0.90090341,  0.57162884,  1.        ],
        [ 0.74944867, -1.12546488,  0.56816232,  1.        ],
        [-1.34813006,  1.40583453,  1.14360433,  1.        ],
        [-0.75334888,  0.33617782, -0.64165251,  1.        ],
        [ 0.01084744, -0.5408078 , -1.05416816,  1.        ],
        [-1.34934888, -1.70347813, -1.42161909,  1.        ],
        [ 0.98102331, -1.67025898, -0.72138242,  1.        ],
        [-0.64853088, -1.4576564 , -0.61738688,  1.        ],
        [ 1.16262659, -0.24847926, -1.31762354,  1.        ],
        [-1.16408916,  0.48898592,  0.82815118,  1.        ],
        [-0.26582331, -1.33806746, -1.20669496,  1.        ],
        [ 1.03343231,  0.34282165,  0.13831407,  1.        ],
        [ 0.9273955 ,  1.32610856,  0.37750382,  1.        ],
        

In [104]:
x_t = np.transpose(x) # x transpose
w = np.zeros((4, 1))
y = y.reshape(20, 1)
min_var = x_t @ ((x @ w) - y)
print("x_t shape: ", x_t.shape,
      "\nx shape: ", x.shape,
      "\nw shape: ", w.shape,
      "\ny shape: ", y.shape)
print("x_t @ ((x @ w) - y): \n", min_var)

x_t shape:  (4, 20) 
x shape:  (20, 4) 
w shape:  (4, 1) 
y shape:  (20, 1)
x_t @ ((x @ w) - y): 
 [[-1.72584963e+01]
 [-8.59220897e+00]
 [-4.09063908e+00]
 [ 4.44089210e-15]]


In [105]:
alpha = 0.1

for i in range(0,5000):
    # w = (2/20)*(w - alpha * (x_t @ ((x @ w) - y)))
    predictions = x @ w
    error = predictions - y
    gradient = (2 / 20) * (x_t @ error) # equivalent to partial derivative; 20 = number of datapoints
    w = w - alpha * gradient # update weights

print(w)

[[ 8.55964629e-01]
 [ 4.79959332e-01]
 [-8.29427362e-02]
 [-2.39253062e-16]]


# Reverse calculate for true values of m1, m2, m3, b
x_mean = np.mean(mat[:, 0:3], axis=0)
x_std = np.std(mat[:, 0:3], axis=0)
y_mean = np.mean(mat[:, 3])
y_std = np.std(mat[:, 3])
stdY_over_stdX = y_std / x_std
# stdY_over_stdX = np.std(mat[:,3], axis=0)  / np.std(mat[:, 0:3], axis=0)
true_w = w[0:3].reshape(1,3) * stdY_over_stdX

temp = np.sum(w[0:3] * stdY_over_stdX * x_mean)
true_b = y_mean + (y_std * w[3]) - temp
true_w, true_b

In [163]:
# Obtain mean and standard deviation
x_mean = np.mean(mat[:, 0:3], axis=0)  # Shape: (3,)
x_std = np.std(mat[:, 0:3], axis=0)    # Shape: (3,)
y_mean = np.mean(mat[:, 3])            # Scalar
y_std = np.std(mat[:, 3])              # Scalar

w_standardized = w[0:3].flatten()      # The 3 feature weights (m1, m2, m3) # Why do we do flatten?
b_standardized = w[3][0]               # The intercept weight (b)

# Reverse calculate true m values (m1, m2, m3)
true_m = w_standardized * (y_std / x_std)

# Reverse calculate true b
adjustment = np.sum(true_m * x_mean) # true_m encapsulates w_standardized * (y_std / x_std)
true_b = y_mean + (y_std * b_standardized) - adjustment

print("True Weights (m1, m2, m3):", true_m)
print("True Intercept (b):", true_b)

--- Unstandardized (Real-World) Parameters ---
True Weights (m1, m2, m3): [[ 0.0552441   0.30113903  0.1571238 ]
 [ 0.03097666  0.16885568  0.08810298]
 [-0.00535314 -0.02918029 -0.01522525]]
True Intercept (b): -18.043871530467136


In [99]:
# 2 variables: output should match the numbers output by LinearRegression class (above)
class myLinearModel:
    def __init__(self): #initialize attributes
        self.slope1 = 0.0
        self.intercept = 0.0
        self.alpha = 0.1

    def fit(self, X, y): # self = refers to the slope and intercept in __init__(self)

        # Standardize data
        x_standard = (X - np.mean(X, axis = 0))/np.std(X, axis = 0) # z-score to standardize data
        y_standard = (y - np.mean(y, axis = 0))/np.std(y, axis = 0)

        # Normal distribution
        x_bias = x_standard.reshape(-1, 1)
        x_bias = np.c_[x_bias, np.ones(len(x_bias))] # Add column of 1's
        x_t = np.transpose(x_bias) # x transpose
        w = np.zeros((2, 1))
        y_bias = y_standard.reshape(-1, 1)
        print("X : ", X.shape, "; x_standard: ", x_standard.shape, "; x_bias: ", x_bias.shape, "; x_t: ", x_t.shape, "; w: ", w.shape, "; y_bias: ", y_bias.shape)
        min_var = x_t @ ((x_bias @ w) - y_bias)

        # Gradient descent
        for i in range(5000):
            predictions = x_bias @ w
            error = predictions - y_bias
            gradient = (2 / len(X)) * (x_t @ error) # equivalent to partial derivative; 20 = number of datapoints
            w = w - self.alpha * gradient # update weights
        
        w_standardized, b_standardized = w[0].item(), w[1].item()
        # print("Standardized m1, b:", w)

        # Reverse calculate true values
        # Obtain mean and standard deviation
        x_mean = np.mean(X, axis=0) 
        x_std = np.std(X, axis=0) 
        y_mean = np.mean(y)            
        y_std = np.std(y)              
        
        w_standardized, b_standardized = w
        
        # Reverse calculate true m values (m1, m2, m3)
        true_m = w_standardized * (y_std / x_std)
        
        # Reverse calculate true b
        adjustment = np.sum(true_m * x_mean) # true_m encapsulates w_standardized * (y_std / x_std)
        true_b = y_mean + (y_std * b_standardized) - adjustment
        
        print("True Weights (m1):", true_m)
        print("True Intercept (b):", true_b)

    def score(self, X, y):
        pass
        # return 

In [100]:
x = df.iloc[:10,0].values.reshape(-1,1)
y = df.iloc[:10,-1].values.reshape(-1,1)

myModel = myLinearModel()
myModel.fit(x, y)

X :  (10, 1) ; x_standard:  (10, 1) ; x_bias:  (10, 2) ; x_t:  (2, 10) ; w:  (2, 1) ; y_bias:  (10, 1)
True Weights (m1): [0.04657946]
True Intercept (b): [7.33401842]


In [141]:
# All variables in dataset
class myLinearModel:
    def __init__(self): #initialize attributes
        self.slope1 = 0.0
        self.slope2 = 0.0
        self.slope3 = 0.0
        self.intercept = 0.0
        self.alpha = 0.1

    def fit(self, X, y): # self = refers to the slope and intercept in __init__(self)

        # Standardize data
        x_standard = (X - np.mean(X, axis = 0))/np.std(X, axis = 0) # z-score to standardize data
        y_standard = (y - np.mean(y, axis = 0))/np.std(y, axis = 0)
        
        # Normal distribution
        x_bias = x_standard
        x_bias = np.c_[x_bias, np.ones(len(x_bias))] # Add column of 1's
        x_t = np.transpose(x_bias) # x transpose
        w = np.zeros((4, 1))
        y_bias = y_standard.reshape(len(X), 1)
        print("X : ", X.shape, "; x_standard: ", x_standard.shape, "; x_bias: ", x_bias.shape, "; x_t: ", x_t.shape, "; w: ", w.shape, "; y_bias: ", y_bias.shape)
        min_var = x_t @ ((x_bias @ w) - y_bias)

        # Gradient descent
        for i in range(5000):
            predictions = x_bias @ w
            error = predictions - y_bias
            gradient = (2 / len(X)) * (x_t @ error) # equivalent to partial derivative; 20 = number of datapoints
            w = w - self.alpha * gradient # update weights

        w_standardized, b_standardized = w[0:3].flatten(), w[3].item()
        # print(w_standardized, b_standardized)

        # Reverse calculate true values
        # Obtain mean and standard deviation
        x_mean = np.mean(X, axis=0) 
        x_std = np.std(X, axis=0) 
        y_mean = np.mean(y)            
        y_std = np.std(y)   
                
        # Reverse calculate true m values (m1, m2, m3)
        true_m = w_standardized * (y_std / x_std)
        
        # Reverse calculate true b
        adjustment = np.sum(true_m * x_mean) # true_m encapsulates w_standardized * (y_std / x_std)
        true_b = y_mean + (y_std * b_standardized) - adjustment
        
        print("True Weights (m1, m2, m3):", true_m)
        print("True Intercept (b):", true_b)

    def score(self, X, y):
        pass
        # return 

In [142]:
x = df.iloc[:10,0:3].values
y = df.iloc[:10,-1].values

myModel = myLinearModel()
myModel.fit(x, y)

X :  (10, 3) ; x_standard:  (10, 3) ; x_bias:  (10, 4) ; x_t:  (4, 10) ; w:  (4, 1) ; y_bias:  (10, 1)
True Weights (m1, m2, m3): [ 0.06334544  0.23743789 -0.06710096]
True Intercept (b): 1.8554832371820318
